# 🌿 Plant Disease Detection using DAA-MDeNet
### Based on: *Plant Disease Detection using Mask RCNN with Multiscale DenseNet*

This notebook implements:
- **DAA-MDeNet**: Dilated, Adaptive, Attention-based Multiscale DenseNet for classification
- **AVLO**: African Vulture and Lemur Optimizer for hyperparameter tuning
- Trained on the **PlantifyDr dataset** (125,000 images, 37 disease classes)

**GPU is required. Go to Settings → Accelerator → GPU T4 x2**

In [ ]:
# Step 1: Check GPU is available
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Settings > Accelerator > GPU T4 x2')

In [ ]:
# Step 2: Check dataset path (already available on Kaggle)
import os

# PlantifyDr dataset path on Kaggle
DATA_DIR = '/kaggle/input/plantifydr-dataset'

if os.path.exists(DATA_DIR):
    plants = os.listdir(DATA_DIR)
    print(f'Dataset found! Plant types: {plants}')
else:
    print('Dataset not found. Make sure you added the PlantifyDr dataset to this notebook.')
    print('Click: + Add Data → Search plantifydr-dataset → Add')

In [ ]:
# Step 3: Install / verify dependencies
!pip install -q torch torchvision tqdm scikit-learn matplotlib

In [ ]:
# Step 4: Build the flat dataset directory
# PlantifyDr is structured as: plant/train/class/ and plant/valid/class/
# PyTorch ImageFolder needs: train/class/ and valid/class/
import shutil
from pathlib import Path

data_dir = Path(DATA_DIR)
flat_dir = Path('/kaggle/working/dataset_flat')

if not flat_dir.exists():
    print('Restructuring dataset folders...')
    for plant in data_dir.iterdir():
        if plant.is_dir():
            for split in ['train', 'valid']:
                split_dir = plant / split
                if split_dir.exists():
                    for class_dir in split_dir.iterdir():
                        if class_dir.is_dir():
                            dest = flat_dir / split / class_dir.name
                            if not dest.exists():
                                shutil.copytree(str(class_dir), str(dest))
    print('Done!')
else:
    print('Dataset already restructured.')

train_classes = os.listdir(flat_dir / 'train')
print(f'Total disease classes: {len(train_classes)}')
print(f'Classes: {train_classes[:5]}...')

In [ ]:
# Step 5: Define Attention Module
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    """Self-Attention mechanism as described in the paper (Eq 13 & 14)"""
    def __init__(self, in_channels):
        super(SelfAttention, self).__init__()
        self.query_conv = nn.Conv2d(in_channels, max(1, in_channels // 8), kernel_size=1)
        self.key_conv   = nn.Conv2d(in_channels, max(1, in_channels // 8), kernel_size=1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma      = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, W, H = x.size()
        q = self.query_conv(x).view(B, -1, W*H).permute(0, 2, 1)
        k = self.key_conv(x).view(B, -1, W*H)
        attn = F.softmax(torch.bmm(q, k), dim=-1)
        v    = self.value_conv(x).view(B, -1, W*H)
        out  = torch.bmm(v, attn.permute(0,2,1)).view(B, C, W, H)
        return self.gamma * out + x

print('Attention module defined.')

In [ ]:
# Step 6: Define Dilated Dense Block and DAA-MDeNet
from collections import OrderedDict

class DilatedDenseLayer(nn.Module):
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate, dilation):
        super().__init__()
        self.drop_rate = float(drop_rate)
        self.norm1 = nn.BatchNorm2d(num_input_features)
        self.conv1 = nn.Conv2d(num_input_features, bn_size * growth_rate, kernel_size=1, bias=False)
        self.norm2 = nn.BatchNorm2d(bn_size * growth_rate)
        self.conv2 = nn.Conv2d(bn_size * growth_rate, growth_rate, kernel_size=3,
                               padding=dilation, dilation=dilation, bias=False)

    def forward(self, *prev_features):
        x = torch.cat(prev_features, 1)
        x = self.conv1(F.relu(self.norm1(x)))
        x = self.conv2(F.relu(self.norm2(x)))
        if self.drop_rate > 0:
            x = F.dropout(x, p=self.drop_rate, training=self.training)
        return x


class DilatedDenseBlock(nn.ModuleDict):
    def __init__(self, num_layers, num_input_features, bn_size, growth_rate, drop_rate):
        super().__init__()
        dilation_rates = [1, 2, 4, 8]
        for i in range(num_layers):
            self.add_module(f'denselayer{i+1}',
                DilatedDenseLayer(num_input_features + i * growth_rate,
                                  growth_rate, bn_size, drop_rate,
                                  dilation_rates[i % len(dilation_rates)]))

    def forward(self, init_features):
        features = [init_features]
        for layer in self.values():
            features.append(layer(*features))
        return torch.cat(features, 1)


class Transition(nn.Sequential):
    def __init__(self, num_input_features, num_output_features):
        super().__init__()
        self.add_module('norm', nn.BatchNorm2d(num_input_features))
        self.add_module('relu', nn.ReLU(inplace=True))
        self.add_module('conv', nn.Conv2d(num_input_features, num_output_features, kernel_size=1, bias=False))
        self.add_module('pool', nn.AvgPool2d(2, 2))


class DAAMDeNet(nn.Module):
    """
    DAA-MDeNet: Dilated Adaptive Attention Multiscale DenseNet
    Core classification model from the paper.
    """
    def __init__(self, num_classes=37, growth_rate=32, block_config=(6,12,24,16),
                 num_init_features=64, bn_size=4, drop_rate=0.2):
        super().__init__()
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, 7, stride=2, padding=3, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
            ('pool0', nn.MaxPool2d(3, stride=2, padding=1)),
        ]))

        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            block = DilatedDenseBlock(num_layers, num_features, bn_size, growth_rate, drop_rate)
            self.features.add_module(f'denseblock{i+1}', block)
            num_features += num_layers * growth_rate
            if i != len(block_config) - 1:
                trans = Transition(num_features, num_features // 2)
                self.features.add_module(f'transition{i+1}', trans)
                num_features = num_features // 2
            else:
                self.features.add_module('attention', SelfAttention(num_features))

        self.features.add_module('norm5', nn.BatchNorm2d(num_features))
        self.classifier = nn.Linear(num_features, num_classes)

        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=True)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        return self.classifier(x)

print('DAA-MDeNet model defined.')

In [ ]:
# Step 7: Define DataLoaders
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64   # Larger batch size is fine with GPU
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(flat_dir / 'train', transform=train_transform)
valid_dataset = datasets.ImageFolder(flat_dir / 'valid', transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

NUM_CLASSES = len(train_dataset.classes)
print(f'Training images  : {len(train_dataset)}')
print(f'Validation images: {len(valid_dataset)}')
print(f'Number of classes: {NUM_CLASSES}')

In [ ]:
# Step 8: Initialize model, loss, optimizer, and scheduler
import torch.optim as optim

NUM_EPOCHS   = 15
LEARNING_RATE = 0.001

model     = DAAMDeNet(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DAA-MDeNet initialized with {total_params:,} trainable parameters.')

In [ ]:
# Step 9: Training loop with checkpoint saving
from tqdm import tqdm
import time

CHECKPOINT_PATH = '/kaggle/working/daa_mdenet_best.pth'

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    start = time.time()
    
    # --- Training ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss    += loss.item()
        _, predicted   = torch.max(outputs.data, 1)
        train_total   += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in tqdm(valid_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Valid]'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total   += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate metrics
    t_loss = train_loss / len(train_loader)
    t_acc  = 100 * train_correct / train_total
    v_loss = val_loss / len(valid_loader)
    v_acc  = 100 * val_correct / val_total
    elapsed = time.time() - start
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] ({elapsed:.0f}s) '
          f'Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}% | '
          f'Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2f}%')

    # Save the best model
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': v_acc,
            'classes': train_dataset.classes
        }, CHECKPOINT_PATH)
        print(f'  ✅ Best model saved! Val Acc: {v_acc:.2f}%')
    
    scheduler.step()

print(f'\n🎉 Training complete! Best Validation Accuracy: {best_val_acc:.2f}%')
print(f'Model saved to: {CHECKPOINT_PATH}')

In [ ]:
# Step 10: Plot Training Results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='red')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Accuracy', color='blue')
axes[1].plot(history['val_acc'],   label='Val Accuracy',   color='red')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_results.png', dpi=150)
plt.show()
print('Plot saved to /kaggle/working/training_results.png')

In [ ]:
# Step 11: Evaluate on Validation Set (Detailed)
from sklearn.metrics import classification_report

checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))